# **RAG AND AGENTIC AI**

In [11]:
!pip install -q faiss-cpu PyPDF2 sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.0 MB/s eta 0:00:00


In [13]:
import faiss #to convert vectors into database
import numpy as np
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
from google.colab import files

In [14]:
#embedding model-converts text into vectors for search
embed_model=SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
#Local LLM for answer generation - loaded directly(no pipeline() wrapper needed)
llm_name="google/flan-t5-base"
tokenizer=AutoTokenizer.from_pretrained(llm_name)
llm_model=AutoModelForSeq2SeqLM.from_pretrained(llm_name)
print("Models loaded successfully")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Models loaded successfully


In [16]:
uploaded = files.upload()   # Select a PDF
pdf_paths = list(uploaded.keys())
print("Uploaded:", pdf_paths)

Saving Basketball AFFILIATION REQUEST LETTER.pdf to Basketball AFFILIATION REQUEST LETTER.pdf
Uploaded: ['Basketball AFFILIATION REQUEST LETTER.pdf']


In [17]:
def extract_text_from_pdfs(paths): #text extraction
    all_text = []
    for path in paths:
        reader = PdfReader(path)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                all_text.append({"source": path, "page": page_num + 1, "text": text})
    return all_text

raw_docs = extract_text_from_pdfs(pdf_paths)
print(f"Extracted {len(raw_docs)} pages.")

Extracted 1 pages.


In [18]:
def chunk_text(docs, chunk_size=500, overlap=50):
    chunks = []
    for doc in docs:
        text = doc["text"]
        start = 0
        while start < len(text):
            chunk = text[start:start + chunk_size]
            chunks.append({"source": doc["source"], "page": doc["page"], "text": chunk})
            start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_docs)
print(f"Created {len(chunks)} chunks.")

Created 2 chunks.


In [20]:
texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)#IndexFlatL2 is the simplest FAISS index type — it does an exact, brute-force comparison against every vector
index.add(embeddings)

print(f"✅ FAISS index built with {index.ntotal} vectors.")


def retrieve_relevant_chunks(query, top_k=3):
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    results = []
    for idx in indices[0]:
        if idx < len(chunks):
            results.append(chunks[idx])
    return results


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built with 2 vectors.


In [21]:
def build_prompt(query, retrieved_chunks):
    context = "\n".join(c["text"] for c in retrieved_chunks)
    prompt = f"Answer the question using only the context.\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"
    return prompt

def ask_document_assistant(query, top_k=3):
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm_model.generate(**inputs, max_new_tokens=200)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, retrieved

In [22]:
class DocumentAgent:
    """
    A minimal 'agent' that:
    1. Remembers conversation history (memory)
    2. Decides to retrieve context before answering (tool use)
    3. Generates a grounded answer (reasoning + action)
    """
    def __init__(self):
        self.history = []

    def ask(self, query):
        answer, sources = ask_document_assistant(query)
        self.history.append({"query": query, "answer": answer})

        print("🧠 Question:", query)
        print("📄 Answer:", answer)
        print("🔍 Sources:")
        seen = set()
        for s in sources:
            key = (s["source"], s["page"])
            if key not in seen:
                print(f"   - {s['source']} (page {s['page']})")
                seen.add(key)
        print("-" * 60)
        return answer

agent = DocumentAgent()

In [23]:
agent.ask("What is this document about?")
agent.ask("Summarize the key points.")

🧠 Question: What is this document about?
📄 Answer: Sports
🔍 Sources:
   - Basketball AFFILIATION REQUEST LETTER.pdf (page 1)
------------------------------------------------------------
🧠 Question: Summarize the key points.
📄 Answer: The Secretary Ernakulam District Basket ball Association is interested to affiliate its club under Ernakulam District Basket ball Association. Kindly let us know the procedure and formalities required for the affiliation process. Looking forward to your guidance and support.
🔍 Sources:
   - Basketball AFFILIATION REQUEST LETTER.pdf (page 1)
------------------------------------------------------------


'The Secretary Ernakulam District Basket ball Association is interested to affiliate its club under Ernakulam District Basket ball Association. Kindly let us know the procedure and formalities required for the affiliation process. Looking forward to your guidance and support.'